# DADA-2000 original — Phase 4 / **train the T2 arms and evaluate them**

Plan: `.project/plans/katvad-dada-original-phase2-t2.md` **§6.2 (pre-registration)** · Corpus: Gate W PASSED 2026-09-16 at W=20 / hop 8.

**Everything measurable about this phase was fixed before the first step ran.** Read plan §6.2
before changing a flag here; a criterion chosen after seeing a delta is not a criterion (C14).

| | metric | bar |
|---|---|---|
| **headline** | DoTA **zero-shot** `auc_macro` | **> 0.6408** |
| attribution | Δ(KIP on − off), paired over 3 seeds | t-interval excluding zero |
| in-domain | T2 `auc_macro` over its **798** two-class windows | reported beside the **0.5983** probe ceiling |
| sanity | T2 micro AUC | only ever beside the **0.7037** clip oracle (C12) |

**Two flags every arm carries, and why** (plan §6.2.1):

* `model.score_head_kernel=3` — kernel 9 spans 45 % of a 20-frame window; a head whose kernel
  spans the clip is a clip classifier (**C27**).
* `loss.mil_topk_pct=5` → k = 4 — the default 16 gives `k = max(1, 20 // 16) = **1**`, i.e.
  `L_MIL` collapses to a plain max with one supervised frame per bag per step. The bag shrank,
  not the formula: MSAD's median clip is 86 frames, where the same pct gives k = 5. **A declared
  deviation**, recorded in every manifest, never tuned against a result.

> **KIP-on is blocked until §3 runs.** `L_KIP_rec` needs `cache/flow/v1/DADA2000_orig`, the CLIP
> pass deleted its frames, so the flow targets cost a second trip through the archive — **train
> ids only** (1,491 sources, flow is train-only). The **KIP-off trunk in §5 needs none of it**
> and is what is trainable today.

> **Do not pass `--set model.*` or `--set kip.*` to `core.evaluate`.** Since **C34** the
> checkpoint defines the architecture (`core/inference.py:adopt_checkpoint_architecture`) and a
> contradicting flag **raises**. The older `colab/DoTA_ncc/*` scripts still carry
> `--set kip.enabled=false` at eval time; they predate that fix. Do not copy them.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no checkout at {REPO}'
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

from core import constants  # noqa: E402

DATASET = constants.DADA_ORIGIN_DATASET
T2 = constants.DATA_ROOT / DATASET                      # the Gate-W corpus
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET           # per SOURCE clip, W-independent
FLOW_DIR = constants.FLOW_CACHE_DIR / DATASET           # written by section 3 (KIP-on only)
KNN_CACHE = constants.KNN_CACHE_DIR / DATASET / constants.KNN_CACHE_FILENAME
RUNS = constants.OUTPUT_ROOT / f'{DATASET}_phase4'      # every arm lands under here
ORIG = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
P4 = Path('/content/p4')                                # VM-local scratch, never Drive
P4.mkdir(parents=True, exist_ok=True)

# Held-out benchmark. NEVER trained on: every comparison this project owns is
# defined by DoTA being unseen. Built by the earlier campaign; section 1 checks it.
DOTA_DATA = constants.DATA_ROOT / 'DoTA' / 'labels_s8'
DOTA_CLIP = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'
DOTA_BAR = 0.6408     # the bar a new training corpus must beat (CLAUDE.md 14.5.1)

# --- the pre-registered arm matrix (plan 6.2) --------------------------------
SEEDS = (2024, 2025, 2026)
# Package values, measured into core/constants.py by Gate W -- restated nowhere.
KERNEL = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL   # 3  (C27)
TOPK_PCT = constants.DADA_ORIGIN_MIL_TOPK_PCT      # 5  -> k = 4 (the declared deviation)
# The horizon is in STEPS; epochs are only how you spell them, and the spelling is
# NOT transferable. Measured 2026-09-16: MSAD's train split is 480 clips / 120
# abnormal -> a DVS epoch of 240 items -> **4 steps/epoch**, so its 125 epochs were
# **500 optimizer steps** (TAD's campaign ran 504). T2 has 3,242 abnormal train
# windows -> 6,484 DVS items -> **102 steps/epoch**, so the same 125 epochs would be
# **12,750 steps -- 25x anything this project has ever run**.
# num_epochs also sets the LR-SCHEDULE horizon, so it must be the INTENDED total;
# --stop-after-epochs caps one invocation without touching that schedule.
NUM_EPOCHS = 20                # = 2,040 steps at 102 steps/epoch (~4x the usual 500)
CHUNK_EPOCHS = 5               # epochs per invocation, so a disconnect costs <= 5

print(f'dataset {DATASET} | kernel {KERNEL} | mil_topk_pct {TOPK_PCT} '
      f'(k = {max(1, constants.DADA_ORIGIN_WINDOW_LENGTH // TOPK_PCT)} '
      f'at W = {constants.DADA_ORIGIN_WINDOW_LENGTH})')
for name, path in (('corpus', T2), ('clip', CLIP_DIR), ('flow', FLOW_DIR),
                   ('knn', KNN_CACHE), ('runs', RUNS),
                   ('DoTA data', DOTA_DATA), ('DoTA clip', DOTA_CLIP)):
    print(f'  {name:10s} {path}   {"OK" if path.exists() else "MISSING"}')

In [ ]:
%%bash
# transformers is PINNED, and not as housekeeping: core/models/clip_text.py reaches into
# three transformers internals (`self.model.text_model` plus two private mask helpers,
# lesson C7). Colab's Python-3.13 runtime ships transformers v5, where CLIPTextModel has
# NO nested .text_model, and training dies ~30 s in -- after the config has been written,
# so the run dir looks started. techContext pins transformers==4.56.*; torch is NOT pinned
# here because torch 2.4 has no cp313 wheel and the feature caches were built on Colab's
# own torch anyway. Restart the runtime after this cell if pip printed a downgrade.
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

### 0.1 Smoke-test the text tower — one second, no download

`SoftPromptCLIPTextModel` is the only place in this pipeline that depends on transformers'
*internal* module layout, so it is the only place a stack drift shows up — and it shows up
**after** `core.train` has resolved the dataset, built the model, downloaded CLIP and written
`config.yaml`, i.e. late enough to look like a training bug. This cell runs that exact forward
pass on a 1-layer random-weight model instead.

In [ ]:
import torch
import transformers
from transformers import CLIPTextConfig, CLIPTextModel

from core.models.clip_text import SoftPromptCLIPTextModel

print(f'python {sys.version.split()[0]} | torch {torch.__version__} | '
      f'transformers {transformers.__version__}')

try:
    tiny = CLIPTextModel(CLIPTextConfig(
        vocab_size=64, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
        num_attention_heads=2, max_position_embeddings=77, eos_token_id=2))
    encoder = SoftPromptCLIPTextModel(clip_model=tiny, num_soft_prompts=4)
    ids = torch.tensor([[0, 5, 6, 2]])
    pooled = encoder(input_ids=ids, attention_mask=torch.ones_like(ids)).pooler_output
    print(f'text tower OK -- pooler_output {tuple(pooled.shape)}')
except AttributeError as exc:
    raise SystemExit(
        f'The text tower does not run on this stack: {exc}\n'
        f'  transformers {transformers.__version__} -- the pipeline needs 4.56.*, where '
        'CLIPTextModel still exposes the nested .text_model submodule that\n'
        '  core/models/clip_text.py:115 reads. transformers v5 flattened it.\n'
        '  Fix: run the cell above, then Runtime -> Restart session, then re-run from 0.\n'
        '  Do NOT "fix" this by porting clip_text.py mid-campaign: it sits on the score\n'
        '  path, an untested port changes z^t silently, and every arm would measure it '
        '(C7, C14).') from exc

### 0.2 Stage the caches on local disk — this is most of the wall clock

`core/train.py` has **no `DataLoader`**: the loop indexes the dataset directly
(`samples = [self.dataset[i] for i in indices]`), serially, in the main process, so that
`--resume` reproduces a run exactly. `data.num_workers` is decoration. One DVS item reads the
anchor clip plus, 30 % of the time (`theta = 0.7`), one to four spliced fillers — **~1.75 `.npy`
reads per item, ~112 per step of 64** — and `_log_metrics` opens `metrics.jsonl` **once per
batch**. Point that at a Google Drive FUSE mount and every one of those is a network round trip.

**The whole CLIP cache is 156 MiB** (1,861 clips × ~43 sampled rows × 512 × float32). Copying it
to the VM's NVMe changes no number — the copy is bit-identical and the transform is untouched
(**C2**) — and removes the per-step round trips.

In [ ]:
import shutil
import time

STAGE = P4 / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)


def stage(src, name):
    # Copy a Drive path to VM-local disk once; return the local path.
    if src is None or not src.exists():
        return src
    dst = STAGE / name
    if dst.exists():
        print(f'  {name:12s} already staged')
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    files = [f for f in dst.rglob('*') if f.is_file()] if dst.is_dir() else [dst]
    mib = sum(f.stat().st_size for f in files) / 2**20
    print(f'  {name:12s} {len(files):5d} files, {mib:7.1f} MiB in {time.time() - t0:5.1f}s')
    return dst


print('staging to VM-local NVMe (bit-identical copies; C2 untouched):')
L_CLIP = stage(CLIP_DIR, 'clip')
L_T2 = stage(T2, 'corpus')
L_KNN = stage(KNN_CACHE, 'knn_cache.npz')
L_DOTA_CLIP = stage(DOTA_CLIP, 'dota_clip')
L_DOTA_DATA = stage(DOTA_DATA, 'dota_data')
L_FLOW = stage(FLOW_DIR, 'flow') if FLOW_DIR.exists() else FLOW_DIR

## 1. Preflight — the corpus must be the one Gate W passed

Training on a corpus that quietly rebuilt at a different geometry is how an arm becomes
unattributable. This cell refuses to continue unless the data dir on Drive **is** the W=20
build, and it reads the exclusion ids out of that build rather than from a runbook.

In [ ]:
import json
import subprocess

ENV = dict(os.environ)

# --- 1.1 the corpus is the geometry Gate W passed ------------------------
windows = json.loads((T2 / constants.WINDOWS_FILENAME).read_text())
meta = json.loads((T2 / constants.META_FILENAME).read_text())
labels_train = json.loads((T2 / constants.LABELS_TRAIN_FILENAME).read_text())
frame_test = json.loads((T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())

lengths = {w['end'] - w['start'] for w in windows.values()}
assert lengths == {constants.DADA_ORIGIN_WINDOW_LENGTH}, (
    f'corpus on disk has windows of {sorted(lengths)}, but this notebook and '
    f'core/constants.py are pre-registered for W={constants.DADA_ORIGIN_WINDOW_LENGTH}. '
    'Re-run phase_2.ipynb, or you are about to train on a corpus nothing gated.')
two_class = sum(1 for v in frame_test.values() if 0 < sum(v) < len(v))
print(f'corpus  : {len(windows)} windows (train {len(labels_train)}, test {len(frame_test)}), '
      f'{two_class} two-class test windows')
assert two_class >= 300, f'{two_class} two-class windows -- Gate W bar is 300'

# --- 1.2 the feature cache covers every source the corpus names ----------
train_ids = (T2 / 'train_ids.txt').read_text(encoding='utf-8').split()
test_ids = (T2 / 'test_ids.txt').read_text(encoding='utf-8').split()
cached = {p.stem for p in CLIP_DIR.glob('*.npy')}
missing = (set(train_ids) | set(test_ids)) - cached
assert not missing, f'{len(missing)} sources have no .npy: {sorted(missing)[:5]}'
print(f'features: {len(cached)} cached | {len(train_ids)} train + {len(test_ids)} test sources')

# --- 1.3 the exclusion list comes from THIS build, not from a runbook ----
# Abnormal sources whose windows hold no positive frame read as genuine normal
# clips (DADA_SETUP.md 5.1). The ids CHANGE with the geometry -- at W=16 they were
# t10_v100 / t48_v056 -- so they are derived here, never pasted.
# Same rule as core/eda/labels.py:vanished_windows -- the unit is the SOURCE clip,
# not the window: a window of an abnormal clip holding no positive frame is a
# correct negative, and producing them is the point of re-sharding. The defect is
# an abnormal source NONE of whose windows carries the anomaly.
by_source = {}
for wid, lab in frame_test.items():
    if meta[wid].get('source_is_abnormal', True):
        by_source.setdefault(meta[wid]['source'], []).append(sum(lab))
VANISHED = sorted(src for src, sums in by_source.items() if max(sums) == 0)
print(f'vanished-window sources in TEST: {VANISHED or "none"}  '
      f'(auc_macro already excludes single-class windows; they matter to micro only)')

# --- 1.4 the held-out benchmark is present and is NOT in the training set
assert DOTA_DATA.is_file() or DOTA_DATA.is_dir(), (
    f'{DOTA_DATA} missing -- build it with core.data.dota (see colab/DoTA_ncc/) before '
    'section 6; the zero-shot number is the headline of this phase')
n_dota = len(list(DOTA_CLIP.glob('*.npy'))) if DOTA_CLIP.is_dir() else 0
print(f'DoTA    : {n_dota} cached clips at {DOTA_CLIP}')
assert not (set(train_ids) & {p.stem for p in DOTA_CLIP.glob("*.npy")}), \
    'a DoTA id appears in the T2 training split -- the held-out set is contaminated'

# --- 1.5 what can be trained right now -----------------------------------
FLOW_READY = FLOW_DIR.is_dir() and len({p.stem for p in FLOW_DIR.glob('*.npy')
                                        if not p.name.endswith('.stats.npy')}) >= len(train_ids)
print('\nKIP-off arms : trainable now')
print(f'KIP-on arms  : {"trainable" if FLOW_READY else "BLOCKED -- run section 3 first"}')

## 2. The KNN cache for DVS

`L_dvs` splices an anchor clip with neighbours; the KNN cache is what makes those neighbours
appearance-matched instead of random. Built once, cheap, CPU.

`faiss` must be single-threaded once torch is imported or it segfaults inside `IndexFlat.search`
— `core/data/knn_cache.py:34-37` does that for you (lesson **C1**).

In [ ]:
def run(cmd, cwd=REPO, capture=False):
    """Run a child; make its failure legible instead of a bare CalledProcessError."""
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}'
                           + ('' if capture else '  -- scroll up for the child output'))
    if capture and proc.stdout:
        print(proc.stdout.rstrip())


if KNN_CACHE.exists():
    print(f'reusing {KNN_CACHE}')
else:
    KNN_CACHE.parent.mkdir(parents=True, exist_ok=True)
    run([sys.executable, '-m', 'core.data.knn_cache',
         '--data-dir', T2, '--dataset', DATASET,
         '--clip-dir', CLIP_DIR, '--output', KNN_CACHE])
    print(f'wrote {KNN_CACHE}')

# Section 0.2 ran before this file existed on a first pass, so re-stage it. It is
# loaded once at startup, not per step, but a Drive path here would be the only
# one left in the training command.
L_KNN = stage(KNN_CACHE, 'knn_cache.npz')

## 3. RAFT flow targets — **only needed for KIP-on**

`L_KIP_rec` regresses cached optical-flow statistics, which exist nowhere yet: `phase_2.ipynb`
deleted every frame after CLIP encoding. This section puts them back one shard at a time, runs
RAFT, and deletes again — the same extract → encode → delete loop, with two differences:

* **train ids only.** Flow is train-time evidence and never touches the inference path
  (`core/data/dataset_files.py:36`), so this covers **`train_ids.txt`**, not the whole corpus.
* **the flow target has no spatial content** — 23 frame-global scalars lifted to 256-d by a
  fixed seeded projection (`core/flow/raft_extract.py:52-81`). Never claim KIP localizes
  anything spatially on the strength of it.

Skip this entire section if you are running the KIP-off trunk only.

In [ ]:
RUN_FLOW = False      # <-- flip to True to spend the archive pass

if not RUN_FLOW:
    print('skipped. KIP-off arms need nothing from here; KIP-on arms will refuse to start.')
else:
    import shutil

    from core.data import dada_origin

    rows = dada_origin.parse_annotation(ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME)
    wanted = set(train_ids)
    shard_rows = [r for r in rows if r.video_id in wanted]
    print(f'{len(shard_rows)} train sources need flow (of {len(rows)} annotated)')

    SHARD = 150
    IMAGES, ROOT = constants.DADA_ORIGIN_IMAGES_SUBDIR, constants.DADA_ORIGIN_ROOT_DIRNAME
    FLOW_DIR.mkdir(parents=True, exist_ok=True)

    for k in range(0, len(shard_rows), SHARD):
        shard = shard_rows[k:k + SHARD]
        tag = f'{k // SHARD:03d}'
        if all((FLOW_DIR / f'{r.video_id}.npy').exists() for r in shard):
            print(f'flow shard {tag}: already extracted, skipping')
            continue
        frames, farm = P4 / 'frames', P4 / f'farm_{tag}'
        shutil.rmtree(frames, ignore_errors=True)
        shutil.rmtree(farm, ignore_errors=True)
        patterns = [f'{ROOT}/{r.type_id}/{r.video:03d}/{IMAGES}/*' for r in shard]
        free = shutil.disk_usage('/content').free / 2**30
        print(f'--- flow shard {tag}: {len(shard)} clips, {free:.1f} GiB free')
        run(['7z', 'x', f'{ROOT}.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns], cwd=ORIG)
        # --census-only builds the symlink farm of globally unique ids (C26); the
        # census itself is already on Drive from phase_2 and is not rewritten here.
        run([sys.executable, '-m', 'core.data.dada_origin',
             '--annotation', ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME,
             '--frames-dir', frames, '--flat-frames-dir', farm, '--census-only',
             '--counts-out', P4 / f'census_{tag}.json'], capture=True)
        run([sys.executable, '-m', 'core.flow.raft_extract',
             '--frames-dir', farm, '--dataset', DATASET,
             '--ids-file', T2 / 'train_ids.txt',
             '--cache-root', constants.FLOW_CACHE_DIR,
             '--stride', constants.FRAME_STRIDE, '--batch-size', '8', '--device', 'auto'])
        shutil.rmtree(frames, ignore_errors=True)
        shutil.rmtree(farm, ignore_errors=True)

    have = {p.stem for p in FLOW_DIR.glob('*.npy') if not p.name.endswith('.stats.npy')}
    print(f'\nflow cache: {len(have)}/{len(train_ids)} train sources at {FLOW_DIR}')
    FLOW_READY = len(have) >= len(train_ids)

## 4. Stage 1 — KIP-only warm-up (**KIP-on arms only**)

Stage 1 trains `L_KIP_rec` / `L_KIP_align` / `L_kin` alone; stage 2 warm-starts from its weights
with `--init-weights`. `--resume` and `--init-weights` are mutually exclusive by design: one
continues a run, the other starts a new one.

**On `main` there is no `kip.gate_type`.** The frozen 321-parameter gate MLP is the only gate and
it never receives a gradient (`floor().long()` on the score path, lesson **C24**), so every
KIP-on arm here is, in operation, a **fixed ~50 % channel shift**. Do not write "motion-gated"
of these runs.

In [ ]:
def train_cmd(out_dir, *, seed, stage, kip_on, init_weights=None):
    """One arm's command line. Every pre-registered flag is here, not in prose."""
    cmd = [sys.executable, '-m', 'core.train',
           '--set', f'train.stage={stage}',
           '--set', f'train.seed={seed}',
           '--set', f'train.num_epochs={NUM_EPOCHS}',
           '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           # --- the two pre-registered flags (plan 6.2.1) ---
           '--set', f'model.score_head_kernel={KERNEL}',
           '--set', f'loss.mil_topk_pct={TOPK_PCT}',
           '--data-dir', L_T2, '--clip-dir', L_CLIP,
           '--knn-cache', L_KNN,
           '--output-dir', out_dir]
    if kip_on:
        cmd += ['--flow-dir', L_FLOW]
    else:
        cmd += ['--set', 'kip.enabled=false']
    if init_weights is not None:
        cmd += ['--init-weights', init_weights]
    return cmd


def stage1_dir(seed):
    return RUNS / f's{seed}' / 'stage1'


def local_dir(drive_dir):
    """The VM-local twin of a Drive run dir: metrics.jsonl is appended EVERY batch."""
    return P4 / 'runs' / drive_dir.relative_to(RUNS)


def sync_to_drive(local, drive):
    """Copy a finished chunk's artifacts to Drive. Small: one ckpt + two text files."""
    drive.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        src = local / name
        if src.exists():
            shutil.copy2(src, drive / name)


def restore_from_drive(drive, local):
    """Bring a previous session's chunk back, so --resume can continue it."""
    if not drive.exists():
        return
    local.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        src = drive / name
        if src.exists() and not (local / name).exists():
            shutil.copy2(src, local / name)


def epochs_done(run_dir):
    """Epochs completed, read from metrics.jsonl -- plain text, unlike a checkpoint.

    Deliberately NOT torch.load on the checkpoint: it carries a pickled numpy RNG
    state that has already outlived one environment once (lesson C15).
    """
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return 0
    last = 0
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            last = max(last, json.loads(line)['epoch'])
    return last + 1


def train_arm(drive_out, *, seed, stage, kip_on, init_weights=None):
    """Train one arm to NUM_EPOCHS in CHUNK_EPOCHS-sized invocations.

    Writes to VM-local disk and syncs to Drive after every chunk, because the run
    dir takes one metrics.jsonl append PER BATCH and a FUSE round trip per append.
    A disconnect costs at most CHUNK_EPOCHS, not the arm.
    """
    local = local_dir(drive_out)
    restore_from_drive(drive_out, local)
    while True:
        done = epochs_done(local)
        if done >= NUM_EPOCHS:
            print(f'  {drive_out.name}: {done}/{NUM_EPOCHS} epochs -- done')
            sync_to_drive(local, drive_out)
            return
        cmd = train_cmd(local, seed=seed, stage=stage, kip_on=kip_on,
                        init_weights=None if done else init_weights)
        if done:                       # continue the same run, same LR schedule
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'  {drive_out.name}: epochs {done} -> '
              f'{min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd)
        sync_to_drive(local, drive_out)


for seed in SEEDS:
    out = stage1_dir(seed)
    if epochs_done(local_dir(out)) >= NUM_EPOCHS or epochs_done(out) >= NUM_EPOCHS:
        print(f'seed {seed}: stage 1 already done, skipping')
        continue
    if not FLOW_READY:
        print(f'seed {seed}: SKIPPED -- no flow cache (section 3). KIP-on is blocked.')
        continue
    print(f'=== seed {seed}: stage 1 (KIP warm-up)')
    train_arm(out, seed=seed, stage=1, kip_on=True)

## 5. Stage 2 — the arms

Six runs: `{KIP off, KIP on} x {2024, 2025, 2026}`. **KIP-off needs no flow and no stage 1** —
it is the trunk, and on its own it answers the question that actually gates the phase: *can this
corpus train anything that transfers?*

Each run writes `checkpoint_last.pt` once per epoch (no mid-epoch checkpoints since `cb7e2ac`) and
a `config.yaml` recording every flag above. **`--set train.checkpoint_every_steps` no longer
exists** — the older MSAD runbook still passes it and would die at config parse here.

In [ ]:
def arm_dir(seed, arm):
    return RUNS / f's{seed}' / f'stage2_kip_{arm}'


ARMS = [(seed, arm) for seed in SEEDS for arm in ('off', 'on')]

for seed, arm in ARMS:
    out = arm_dir(seed, arm)
    if epochs_done(local_dir(out)) >= NUM_EPOCHS or epochs_done(out) >= NUM_EPOCHS:
        print(f'seed {seed} kip_{arm}: already trained, skipping')
        continue
    if arm == 'on' and not FLOW_READY:
        print(f'seed {seed} kip_on: BLOCKED -- no flow cache (section 3)')
        continue
    init = local_dir(stage1_dir(seed)) / 'checkpoint_last.pt' if arm == 'on' else None
    if init is not None and not init.exists():
        print(f'seed {seed} kip_on: BLOCKED -- stage 1 has not run')
        continue
    print(f'=== seed {seed}: stage 2, KIP {arm}  '
          f'({NUM_EPOCHS} epochs = {NUM_EPOCHS * 102} steps)')
    train_arm(out, seed=seed, stage=2, kip_on=(arm == 'on'), init_weights=init)

## 6. Evaluation — in-domain T2, then **zero-shot DoTA**

Two protocols, and the difference between them is the data, not a preference:

* **T2 (in-domain)** — 27.2 % of test windows are all-normal, so `--score-norm auto` resolves to
  **`none`**. Sanity only.
* **DoTA (zero-shot)** — 1,394 of 1,397 clips are abnormal, so `auto` resolves to **per-clip
  min-max**, which is the protocol the published 62.60 is defined by (**C8**, **C12**).

**No `--set model.*` and no `--set kip.*` here.** The checkpoint carries its own architecture and
a contradicting flag raises (**C34**) — which is exactly what you want: it means an arm cannot be
scored with the treatment it was trained without.

In [ ]:
def evaluate(ckpt, out_dir, *, dataset, data_dir, clip_dir):
    if (out_dir / 'results.json').exists():
        return json.loads((out_dir / 'results.json').read_text())
    run([sys.executable, '-m', 'core.evaluate',
         '--ckpt', ckpt,
         '--set', f'data.dataset={dataset}',
         '--data-dir', data_dir, '--clip-dir', clip_dir,
         '--output-dir', out_dir, '--save-scores'])
    return json.loads((out_dir / 'results.json').read_text())


EVALS = {}
for seed, arm in ARMS:
    ckpt = local_dir(arm_dir(seed, arm)) / 'checkpoint_last.pt'
    if not ckpt.exists():
        ckpt = arm_dir(seed, arm) / 'checkpoint_last.pt'
    if not ckpt.exists():
        print(f'seed {seed} kip_{arm}: no checkpoint, skipping eval')
        continue
    EVALS[(seed, arm, 'dota')] = evaluate(
        ckpt, RUNS / f's{seed}' / f'eval_dota_kip_{arm}',
        dataset='DoTA', data_dir=L_DOTA_DATA, clip_dir=L_DOTA_CLIP)
    EVALS[(seed, arm, 't2')] = evaluate(
        ckpt, RUNS / f's{seed}' / f'eval_t2_kip_{arm}',
        dataset=DATASET, data_dir=L_T2, clip_dir=L_CLIP)
print(f'\n{len(EVALS)} evaluations on disk')

## 7. The table, the paired delta, and the verdict

`auc_macro` is the headline on both corpora. The micro number appears **only** beside the thing
that explains it: on DoTA the pooling rule, on T2 the 0.7037 clip oracle.

In [ ]:
import statistics

CEILING_T2 = 0.5983     # supervised frame probe on these very windows (plan 6.2.2)
ORACLE_T2 = 0.7037      # constant-score-per-clip, micro
RULER_T2 = 0.5000       # frame-count-only baseline


def cell(res, key):
    return float(res[key]) if res is not None and key in res else float('nan')


print(f'{"seed":>5} {"arm":>4} | {"DoTA macro":>10} {"DoTA micro":>10} {"norm":>7} '
      f'| {"T2 macro":>8} {"T2 micro":>8} {"norm":>5}')
print('-' * 78)
for seed, arm in ARMS:
    d, t = EVALS.get((seed, arm, 'dota')), EVALS.get((seed, arm, 't2'))
    if d is None and t is None:
        continue
    print(f'{seed:>5} {arm:>4} | {cell(d, "auc_macro"):>10.4f} {cell(d, "auc"):>10.4f} '
          f'{d.get("score_norm") if d else "-"!s:>7} '
          f'| {cell(t, "auc_macro"):>8.4f} {cell(t, "auc"):>8.4f} '
          f'{t.get("score_norm") if t else "-"!s:>5}')

print(f'\nbars: DoTA macro > {DOTA_BAR} (headline) | T2 macro vs probe ceiling {CEILING_T2} '
      f'| T2 micro vs clip oracle {ORACLE_T2} | length ruler {RULER_T2}')

# --- the paired delta, over seeds present in BOTH arms ----------------------
def dota_macro(seed, arm):
    return cell(EVALS.get((seed, arm, 'dota')), 'auc_macro')


paired = [dota_macro(s, 'on') - dota_macro(s, 'off') for s in SEEDS
          if (s, 'on', 'dota') in EVALS and (s, 'off', 'dota') in EVALS]
if len(paired) >= 2:
    mean = statistics.mean(paired)
    # t95 for n-1 df; the campaign has always reported n=3, i.e. t = 4.303
    tcrit = {2: 12.706, 3: 4.303}.get(len(paired), 2.776)
    half = tcrit * statistics.stdev(paired) / len(paired) ** 0.5
    print(f'\nDelta(KIP on - off) on DoTA auc_macro, n={len(paired)}: '
          f'{mean:+.4f} +/- {half:.4f}  -> CI [{mean - half:+.4f}, {mean + half:+.4f}]')
    print('  CI includes zero -> KIP is REFUTED on this corpus (plan 6.2.4). That is the '
          'pre-registered expected outcome, not a disappointment.'
          if mean - half <= 0 <= mean + half else
          '  CI excludes zero -> a real effect. Attribute it before claiming it is motion: '
          'the MSAD +0.09 was a temporal smoother (v3/RESULTS_V3_GATE_ATTRIBUTION.md).')
else:
    print('\nno seed has both arms yet -- the paired delta needs KIP-on, which needs section 3')

trunk = [cell(EVALS[(s, 'off', 'dota')], 'auc_macro') for s in SEEDS if (s, 'off', 'dota') in EVALS]
if trunk:
    mean_trunk = statistics.mean(trunk)
    verdict = 'CLEARS' if mean_trunk > DOTA_BAR else 'MISSES'
    print(f'\nKIP-off trunk, DoTA macro: {mean_trunk:.4f} over {len(trunk)} seeds '
          f'-> corpus {verdict} the {DOTA_BAR} bar')
    if mean_trunk <= DOTA_BAR:
        print('  Per plan 6.2.4 the CORPUS is refuted, not KIP: stop and diagnose T2 as '
              'another C14 case rather than trying a KIP variant.')

## 8. Record the campaign (**C17**)

Copy every `results.json` and `config.yaml` off the VM in the same cell that read them, and
record the flags that define what these runs were — `cfg.save_yaml()` writes the config tree
only, so `--init-weights`, the data paths and the git commit live nowhere else.

In [ ]:
import shutil

dest = constants.OUTPUT_ROOT / 'REPORTS' / f'{DATASET}_phase4'
dest.mkdir(parents=True, exist_ok=True)

for (seed, arm, bench), res in EVALS.items():
    tag = f's{seed}_kip_{arm}_{bench}'
    (dest / f'results_{tag}.json').write_text(json.dumps(res, indent=2), encoding='utf-8')
    cfg = arm_dir(seed, arm) / 'config.yaml'
    if cfg.exists():
        shutil.copy2(cfg, dest / f'config_s{seed}_kip_{arm}.yaml')

proc = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True)
commit = proc.stdout.strip() or f'UNKNOWN (git said: {proc.stderr.strip()!r})'

manifest = {
    'phase': 'Phase 4 (T2 arms + evaluation)',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'commit': commit,
    'corpus': {'data_dir': str(T2), 'window_length': constants.DADA_ORIGIN_WINDOW_LENGTH,
               'window_stride': constants.DADA_ORIGIN_WINDOW_STRIDE,
               'train_windows': len(labels_train), 'test_windows': len(frame_test),
               'two_class_test_windows': two_class, 'gate_w': 'PASSED 2026-09-16'},
    'preregistered': {'score_head_kernel': KERNEL, 'mil_topk_pct': TOPK_PCT,
                      'num_epochs': NUM_EPOCHS, 'seeds': list(SEEDS),
                      'dota_bar': DOTA_BAR, 't2_probe_ceiling': CEILING_T2,
                      't2_clip_oracle': ORACLE_T2, 'plan': 'phase2-t2.md 6.2'},
    'caches': {'clip': str(CLIP_DIR), 'flow': str(FLOW_DIR) if FLOW_READY else None,
               'knn': str(KNN_CACHE)},
    'excluded_at_scoring': VANISHED,
    'arms': {f's{seed}_kip_{arm}': {
        'checkpoint': str(arm_dir(seed, arm) / 'checkpoint_last.pt'),
        'trained': (arm_dir(seed, arm) / 'checkpoint_last.pt').exists(),
        'dota_auc_macro': cell(EVALS.get((seed, arm, 'dota')), 'auc_macro'),
        't2_auc_macro': cell(EVALS.get((seed, arm, 't2')), 'auc_macro'),
    } for seed, arm in ARMS},
}
(dest / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('recorded ->', dest)
for p in sorted(dest.iterdir()):
    print(f'   {p.name}  ({p.stat().st_size / 1024:.1f} KiB)')
print(f'\ncommit: {commit}')
print('\nNext: write the numbers up in core/docs/ and NEVER put a T2 micro AUC beside a '
      'published frame-level figure (C8, C8b, C12). If KIP moved anything, attribute it '
      'before naming it -- the MSAD +0.09 was a temporal smoother, not motion.')